# Phase 5 — the column nobody asked for

Three sessions of this project have been spent asking why our Rossmann runs sit
below the private median, and the candidate answers were all about things we
*cannot* do: features the winner had and our vocabulary cannot express, a search
that gives up too early, a budget that is too small.

[0071](../docs/decisions/0071-the-gap-was-not-the-vocabulary-it-was-a-column-nobody-asked-for.md)
measured all three at once, and the largest number was none of them:

| | probe rmspe |
|---|---|
| capability gap — three calendar windows no tool can express | +0.00122 |
| agent gap — the winner's feature set against what our agent built | +0.00220 |
| **`fit_history` emitting a column pair that was never requested** | **+0.01428** |

**The tool defect is six times the agent gap and twelve times the capability
gap.** This notebook is that finding driven by hand: what the pair is, why it is
there, what it looks like on the rows a model is actually served, and the
before/after of removing it.

Nothing here changes the library. The fix is a few lines and it is deliberately
not made yet — see the last section.

*Style, as always: drive the machinery by hand first. Every code cell was
smoke-tested in order; the notebook ships unexecuted.*

In [ ]:
import os
import sys

import numpy as np
import pandas as pd

REPO = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, os.path.join(REPO, "src"))
sys.path.insert(0, os.path.join(REPO, "scripts"))

# The run 0071 measured: the winner's salesmodel rebuilt with real tool calls, on
# v7's copied split. We reuse its split rather than cutting a new one, so every
# number below sits on the same scale as the record's.
RUN = os.path.join(REPO, "_runs", "winner_salesmodel", "A_tools_only")

work = pd.read_parquet(os.path.join(RUN, "data", "work.parquet"))
val = pd.read_parquet(os.path.join(RUN, "data", "val.parquet"))

print(f"work (what history may be fitted on) : {len(work):>7,} rows, "
      f"{work['Date'].min().date()} .. {work['Date'].max().date()}")
print(f"val  (the rows it is scored on)      : {len(val):>7,} rows, "
      f"{val['Date'].min().date()} .. {val['Date'].max().date()}")

## 1. Ask for one column, get four

The winner's `prevquarter_med` is one feature: *the median of this store's sales
over the last quarter*. In our vocabulary that is one `fit_history` call with one
window. Here is the encoder underneath it, driven directly.

In [ ]:
from tabular_agent.encoders import HistoryEncoder

enc = HistoryEncoder(by=["Store"], value="Sales", order="Date", windows=[(91, "median")])
enc.fit(work)

print("asked for : the median of each store's sales over the last 91 days")
print("got       :")
for name in enc.output_columns:
    print(f"    {name}")

Two of those four were requested. `_91_median` is the feature; `_91_count` is the
number of observations behind it, and `HistoryEncoder`'s docstring argues at
length for why a summary must carry its own sample size — a mean over two points
and a mean over ninety are the same number wearing the same name.

The other two were not requested and cannot be declined:

* `Store_Sales_latest` — the most recent observation for this row's group, as of
  the freeze point,
* `Store_Sales_latest_ago` — how far back that observation sits, in days.

The class documents this as *"the pair is the design, not a convenience"*, and
that argument is sound **for the case it was written about**: if you are going to
serve a last-known value, serving it without its age is a carry-forward with the
honesty removed. What the docstring never asks is the prior question — whether a
caller who asked for a *window* wanted a last-known value at all.

In [ ]:
print(f"stored per group : {enc.keep} observations")
print(f"frozen at        : {pd.Timestamp(enc.frozen_at_, unit='D').date()}")
print(f"groups           : {len(enc.history_):,}")

## 2. What the served rows actually get

`fit` stores each group's tail and freezes. Every row after the freeze reads that
stored state — which is what makes the column batch-independent and servable, and
is the whole reason `fit_history` exists where a `groupby().shift()` is refused.

Look at what two stores get for the 45 days after the freeze.

In [ ]:
STORES = [1, 3]
served = val[val["Store"].isin(STORES)].sort_values(["Store", "Date"])
feat = enc.transform(served)
cols = ["Store", "Date", "Sales", *enc.output_columns]
print(feat[cols].head(10).to_string(index=False))

`_91_median` moves with the row — each row averages its own stretch of the past,
which is the property 0050 is about. `_latest_ago` counts up 1, 2, 3, … exactly as
designed: the value is getting older and the column says so.

And `Store_Sales_latest` is **0.0**.

Not missing — zero. The last observation before the freeze was a day on which
this store sold nothing.

In [ ]:
allfeat = enc.transform(val)
per_store_latest = allfeat.groupby("Store")["Store_Sales_latest"].first()
print(f"stores served             : {len(per_store_latest):,}")
print(f"whose _latest is exactly 0: {(per_store_latest == 0).sum():,} "
      f"({(per_store_latest == 0).mean():.1%})")

freeze_day = work[work["Date"] == work["Date"].max()]
print(f"\nthe freeze point is {work['Date'].max().date()}, and on that day:")
print(freeze_day.groupby(["StateHoliday", "Open"]).size().to_string())

The freeze lands on **1 May 2015**, a public holiday, and 1,095 of 1,115 stores
were shut. So for 98% of stores the column reads *"this shop's most recent sales
figure is zero"* for all 45 days it is served.

**This is 0049 in its purest form.** That record is titled *the last observation
is a sample of size one*, and the failure mode it names is exactly this: one draw
decides the column, and a draw can land anywhere — including on the one day of the
quarter when the whole country was closed. The `_ago` column faithfully reports
that the number is 1 to 45 days old. It has no way to report that the number is
meaningless.

Note what is *not* wrong here. Nothing leaked, nothing is inconsistent between
fit and serve, and the encoder answered the question it was asked correctly. The
harness has no guard that could fire, because there is no error.

## 3. The column the model trains on is not the column it is served

This is the part that turns a useless feature into a harmful one.

Training rows do not read the final freeze. `fit_history` freezes **per fold** —
each fold's history stops at the end of its own training prefix — so a training
row's `_latest` comes from that fold's boundary, and the boundaries are scattered
across two and a half years. Compare the two distributions.

In [ ]:
t10 = pd.read_parquet(os.path.join(RUN, "tables", "t10.parquet"),
                      columns=["Date", "Store", "Store_Sales_latest", "Store_Sales_latest_ago"])

def describe(name, s, ago):
    print(f"{name:<28} distinct {s.nunique():>6,}   zero {(s == 0).mean():>6.1%}   "
          f"age {ago.min():.0f}..{ago.max():.0f} days")

describe("training rows (per fold)", t10["Store_Sales_latest"], t10["Store_Sales_latest_ago"].dropna())
describe("served rows (final freeze)", allfeat["Store_Sales_latest"],
         allfeat["Store_Sales_latest_ago"].dropna())

Four thousand distinct values and essentially no zeros at training time; **21
distinct values across 1,115 stores at serving time — one per store, and 1,095 of
them zero.** Ages 1–305 against 1–45.

A gradient-boosted tree fitted on the first row of that table will happily split
on `Store_Sales_latest`, because in training it is a genuinely informative
recent-sales number. At serving time the same split sends almost every row down
the "this store sells nothing" branch. The model is not being lied to about a
value; it is being trained on one column and served another that shares its name.

That is why the ablation is worth 0.014 rather than 0.000. An idle column costs a
boosted tree almost nothing — 0069 measured exactly that, dropping nine pure-noise
columns for 0.0014. This one is not idle. It is *confidently wrong*.

In [ ]:
open_rows = allfeat[(allfeat["Open"] == 1) & (allfeat["Sales"] > 0)]
for name in ("Store_Sales_latest", "Store_Sales_91_median"):
    c = open_rows[[name, "Sales"]].corr().iloc[0, 1]
    print(f"corr({name:<22}, Sales) = {c:>6.3f}   on the rows that get scored")

## 4. The picture

Grey is history, blue is the target over the rows we are graded on, green is the
feature that was asked for, red is the pair member that came with it. The dotted
line is the freeze.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(STORES), 1, figsize=(11, 3.2 * len(STORES)), sharex=True)
freeze = pd.Timestamp(enc.frozen_at_, unit="D")
for ax, store in zip(np.atleast_1d(axes), STORES):
    hist = work[(work["Store"] == store) & (work["Open"] == 1)].sort_values("Date").tail(120)
    fwd = allfeat[(allfeat["Store"] == store) & (allfeat["Open"] == 1)].sort_values("Date")
    ax.plot(hist["Date"], hist["Sales"], color="0.75", lw=1, label="Sales, before the freeze")
    ax.plot(fwd["Date"], fwd["Sales"], color="C0", lw=1.2, label="Sales, the rows we score on")
    ax.plot(fwd["Date"], fwd["Store_Sales_latest"], color="C3", lw=2,
            label="Store_Sales_latest  (one draw, carried forward)")
    ax.plot(fwd["Date"], fwd["Store_Sales_91_median"], color="C2", lw=2,
            label="Store_Sales_91_median  (what was asked for)")
    ax.axvline(freeze, color="k", ls=":", lw=1)
    ax.set_title(f"store {store}")
    ax.set_ylabel("Sales")
np.atleast_1d(axes)[0].legend(loc="upper left", fontsize=8, ncols=2)
plt.tight_layout()
plt.show()

The red line sitting on zero is the column that costs 0.014.

## 5. The second call, keyed finer

The rebuild made a second `fit_history` call — `prevyear_dps_med`, a median by
store × day-of-week × promo over the last year. A finer key means each group's
last observation falls on a different date, so the pathology is milder. It does
not go away.

In [ ]:
enc2 = HistoryEncoder(by=["Store", "DayOfWeek", "Promo"], value="Sales", order="Date",
                      windows=[(364, "median")])
enc2.fit(work)
f2 = enc2.transform(val)

name = "Store_DayOfWeek_Promo_Sales_latest"
ago = f2[f"{name}_ago"]
print(f"{name}")
print(f"  distinct values within one served group : "
      f"{f2.groupby(['Store', 'DayOfWeek', 'Promo'])[name].nunique().max()}")
print(f"  exactly 0                               : {(f2[name] == 0).mean():.1%} of served rows")
print(f"  age in days, median / max               : {ago.median():.0f} / {ago.max():.0f}")

Still one value per group for the whole horizon — that is structural, not a
Rossmann accident: the freeze does not move while the rows do. The zero share is
22% rather than 98%, and the value is a month old at the median.

Both calls contribute a pair, so the table carries four unrequested columns.

## 6. The price, before and after

Everything above is mechanism. This is the measurement, read off the run's own
journal: the finished table, then the *same table with only those four columns
dropped*. Nothing else changes — same rows, same split, same budget.

In [ ]:
import json

journal = json.load(open(os.path.join(RUN, "run_journal.json")))
tables = {t["id"]: t for t in journal["tables"]}

for tid in ("t8", "t9", "t10", "t11"):
    t = tables[tid]
    gain = t.get("probe_vs_source_table")
    tail = f"  ({gain:+.5f} vs parent)" if gain is not None else ""
    print(f"{tid:<4} {t['method']:<10} probe {t['probe_score']:.5f}{tail}")

print("\nthe ablation step:")
print(tables["t11"]["code"])

    with the pair    0.14098
    without it       0.12670

For scale, v7's `t13` — the best table our agent built on this dataset by itself —
probes at 0.13202. **Dropping four columns nobody asked for beats it.**

A difference is not a finding until it has an interval around it, which is what
0059 built `compare.py` for. Recompute it here from the stored probe predictions
rather than quoting the journal's copy.

In [ ]:
from tabular_agent.compare import compare_predictions

with_pair = np.load(os.path.join(RUN, "probe_preds", "t10.npy"))
without = np.load(os.path.join(RUN, "probe_preds", "t11.npy"))

interval = compare_predictions("rmspe", val["Sales"], without, with_pair,
                               order_values=val["Date"])
for k, v in interval.items():
    print(f"  {k:<20} {v}")

    block gains  [+0.0150, +0.0142, +0.0128, +0.0072, +0.0249]
    block sd 0.00576   sign_stable True   5 of 5 blocks favour dropping it

**2.5 block standard deviations with every block agreeing.** That is the first
reading in three sessions that passes 0059's instrument convincingly — for
contrast, both of 0069's feature-selection readings came back `sign_stable: False`
and inside their own noise.

The blocks are worth reading individually and not just as a spread: they do not
decay across the window, which is what 0059 found for the arm Kaggle later
refuted. Sealed, the effect is roughly +0.008 — the same probe→sealed halving 0068
measured for capacity.

## 7. What this is, and what it is not

**It is a tool defect, and it names no dataset.** The general statement:

> The harness cannot express *"summarise this group's recent past"* without also
> adding a column pair the caller may not want and has no way to decline.

Any panel dataset with a freeze point and many rows per key can hit it. What
Rossmann supplies is only the vividness — a national holiday one day before the
freeze.

**It is not new information, and that is its own lesson.** 0049 measured the last
observation as a sample of size one. 0050 found the anchor mattered more than the
history. `HANDOFF.md` has carried *"`_latest` still emitted unconditionally"* as an
open item for weeks. What was missing was the **price on a table worth having** —
every earlier reading put the pair on a bare table where it was one nuisance among
many, and there it was cheap.

**It is n=1.** One dataset, one ablation, one probe. The record this project keeps
breaking is the one about acting on a single reading — 0069 is three days old and
is exactly that mistake. So the fix is *not made here*.

### What comes next, in order

1. **Confirm on a second dataset.** Restaurant's `t7` is on disk and carries the
   same ablation in about an hour. Its spine is ragged and its groups are shops
   rather than stores, so it is a real test rather than a rerun.
2. **Then the few-line change**: let `fit_history` decline the pair. The design
   question is not *whether* but *what the default is* — a caller who asks only for
   `offsets` and no windows may well want it, and 0049's argument that a bare
   carry-forward is dishonest still stands.
3. **Then a record**, saying which of 0049's and 0050's conclusions survive.

### One more thing this exposed, filed separately

The naive version of the winner's holiday windows — roll a per-date flag over
seven days — was **accepted** by 0011's stability check, and it should not have
been. `transform.py` resamples *rows*; with 1,115 stores per date the chance a
sample of 25% drops a date entirely is `0.75 ** 1115`, so `groupby(Date)` is
unchanged and the column looks perfectly stable. **The check tests row-locality,
not batch-independence**, and any column that is a function of the set of distinct
keys passes it silently. That gets its own notebook and its own record: widening
the check changes what *every* existing transform is allowed to do, so what it
would newly refuse has to be measured before it is changed.